<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/NCL_datos_entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Praparar datos de entrenamiento para Nevado de Colima

In [9]:
# Prefijo del área
PREFIJO     = 'NCL'
# Nombre del área
NOMBRE      = 'Nevado_Colima'

# Carpeta de salida en Drive
#RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI/Colima'
RUTA_AOI    = '/content/drive/MyDrive/Colab Data/Naylet/AOI'

# Rutas
RUTA_GEOJSON = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_BBOX   = f'{RUTA_AOI}/{PREFIJO}_bbox.geojson'
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
#GEE_PROJECT = 'ee-nayleths'
GEE_PROJECT = 'ee-vshalisko'

# Año
AÑO = 2020

fecha_inicio_exacto = f'{AÑO}-01-01'
fecha_fin_exacto   = f'{AÑO}-12-31'


# Escala de exportación (m)
ESCALA        = 30

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo o bounding box
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio Nevado de Colima'

# Carpeta de salida en Drive para imágenes GEE
#RUTA_IMAGENES = f'/content/drive/MyDrive/ANP/Landsat'
RUTA_IMAGENES = 'Colab Data NDC 2020'


In [2]:
import ee
import os
import math
import json
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
#from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Mounted at /content/drive
✓ Entorno listo


# Bases de datos con LULC de referencia
1. EC JRC global map of forest types V1 (2020)

2. Dynamic World (Derivado de Sentinel)

3. Copernicus Global Land Cover Layers (2015 en adelante)

In [10]:
# Región GEE desde el GeoJSON generado
gdf = gpd.read_file(RUTA_BBOX)
geojson_str = gdf.to_crs(epsg=CRS_GEO).to_json()
region = ee.FeatureCollection(json.loads(geojson_str)).geometry()


# 'JRC/GFC2020_subtypes/V1'
# 'GOOGLE/DYNAMICWORLD/V1'
# 'COPERNICUS/Landcover/100m/Proba-V-C3/Global/2019'

ref_image_EC = ee.Image('JRC/GFC2020_subtypes/V1')

ref_collection_DW = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
                .filterDate(fecha_inicio_exacto, fecha_fin_exacto)
                .filterBounds(region))

ref_image_DW = ref_collection_DW.first()

In [25]:
ref_image_DW_mode = ref_collection_DW.select('label').reduce(ee.Reducer.mode())
ref_image_DW_label_mode = ref_image_DW_mode.select('label_mode')

In [22]:
# Define list pairs of DW LULC label and color.
CLASS_NAMES = [
    'water',
    'trees',
    'grass',
    'flooded_vegetation',
    'crops',
    'shrub_and_scrub',
    'built',
    'bare',
    'snow_and_ice',
]

VIS_PALETTE = [
    '419bdf',
    '397d49',
    '88b053',
    '7a87c6',
    'e49635',
    'dfc35a',
    'c4281b',
    'a59b8f',
    'b39fe1',
]

# Create an RGB image of the label (most likely class) on [0, 1].
dw_rgb = (
    #ref_image_DW.select('label')
    ref_image_DW_mode.select('label_mode')
    .visualize(min=0, max=8, palette=VIS_PALETTE)
    .divide(255)
)

In [ ]:
m = geemap.Map()
m.set_center(-103.5, 19.5, 11)
m.add_layer(ref_image_EC, {}, 'Land Cover')
m.add_layer(
    dw_rgb,
    {'min': 0, 'max': 1},
    'Dynamic World V1 - label hillshade',
)
m

In [26]:
# Exportar a Google Drive
def exportar(imagen, fuente):
    if imagen is None:
        print(f"Exportación cancelada para '{fuente}': imagen no disponible.")
        return

    nombre_tarea = f'{PREFIJO}_{AÑO}_{fuente}'
    tarea = ee.batch.Export.image.toDrive(
        image          = imagen,
        description    = nombre_tarea,
        folder         = RUTA_IMAGENES,
        fileNamePrefix = nombre_tarea,
        region         = region,
        scale          = ESCALA,
        crs            = f'EPSG:{CRS_UTM}',
        maxPixels      = 1e13
    )
    tarea.start()
    print(f"Tarea iniciada: {nombre_tarea}")

exportar(ref_image_EC,  'ref_EC')
exportar(ref_image_DW_label_mode,  'ref_DW')

Tarea iniciada: NCL_2020_ref_EC
Tarea iniciada: NCL_2020_ref_DW
